In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential,layers
from tensorflow.keras.datasets import mnist


In [2]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [41]:
x_train =x_train.reshape(-1,28, 28, 1)/ 255.0
x_test = x_test.reshape(-1,28, 28, 1)/ 255.0

In [4]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 8.2 MB/s eta 0:00:00


In [5]:
from keras_tuner import RandomSearch

In [9]:
def build_model(hp):
  model = Sequential(
      [
          layers.Dense(units = hp.Int('units1', min_value=32, max_value = 256, step = 32), activation = hp.Choice('activation1', ['sigmoid', 'relu', 'tanh'])),
          layers.Dense(units = hp.Int('units2', min_value=32, max_value = 256, step = 32), activation = hp.Choice('activation2', ['sigmoid', 'relu', 'tanh'])),
          layers.Dense(10, activation = "softmax")
      ]
  )
  model.compile(optimizer = keras.optimizers.Adam(learning_rate = hp.Choice('lr', [0.01, 0.001, 0.0001])),
                loss = keras.losses.SparseCategoricalCrossentropy(),
                metrics = ['accuracy'])
  return model

In [10]:
tuner = RandomSearch(build_model, max_trials = 5, objective = 'val_loss', directory='/mydir')

In [14]:
tuner.results_summary()

Results summary
Results in /mydir/untitled_project
Showing 10 best trials
Objective(name="val_loss", direction="min")


In [15]:
tuner.search(x_train, y_train, epochs = 5, validation_data = (x_test, y_test))

Trial 5 Complete [00h 00m 29s]
val_loss: 0.08479124307632446

Best val_loss So Far: 0.08479124307632446
Total elapsed time: 00h 02m 32s


In [22]:
def build_model2(hp):
  model = Sequential()
  for i in range(hp.Int('layers', min_value = 2, max_value = 6, step = 1)):
    model.add(layers.Dense(units = hp.Int("unit"+str(i), min_value = 32, max_value = 256, step = 32), activation=hp.Choice("acivation"+str(i), ["sigmoid", "relu", "tanh"])))
  model.add(layers.Dense(10, activation = "softmax"))

  model.compile(optimizer = keras.optimizers.Adam(learning_rate = hp.Choice('lr', [0.01, 0.001, 0.0001])), loss = keras.losses.SparseCategoricalCrossentropy(), metrics = ['accuracy'])
  return model

In [23]:
tuner2 = RandomSearch(build_model2, objective = "val_loss", directory = "/mydir2", max_trials = 5)

In [24]:
tuner2.results_summary()

Results summary
Results in /mydir2/untitled_project
Showing 10 best trials
Objective(name="val_loss", direction="min")


In [25]:
tuner2.search(x_train, y_train, epochs = 5, validation_data = (x_test, y_test))

Trial 5 Complete [00h 00m 35s]
val_loss: 0.12345850467681885

Best val_loss So Far: 0.0899386778473854
Total elapsed time: 00h 03m 00s


In [47]:
class CNNBlock(layers.Layer):
  def __init__(self, num_kernels, kernel_size = 3):
    super(CNNBlock, self).__init__()
    self.conv1 = layers.Conv2D(num_kernels, kernel_size, padding = "same")
    self.normalisation = layers.BatchNormalization()
    self.pooling = layers.MaxPooling2D()

  def call(self, input):
    x = self.conv1(input)
    x = self.normalisation(x)
    x = keras.activations.relu(x)
    out = self.pooling(x)
    return out

In [52]:
def build_model3(hp):
  model = Sequential()
  for i in range(0, hp.Int('blocks', min_value = 2, max_value = 4, step = 1)):
    model.add(CNNBlock(num_kernels = hp.Int("kernels" + str(i), min_value = 32, max_value = 512, step = 32), kernel_size= hp.Choice('kernel_Size_' + str(i), [3, 4, 5])))
  model.add(layers.Flatten())
  model.add(layers.Dense(hp.Int('units', min_value = 32, max_value = 512, step = 32), activation = hp.Choice('act', ["relu", "tanh"])))
  model.add(layers.Dense(10, activation = "softmax"))

  model.compile(optimizer = keras.optimizers.Adam(learning_rate = 0.01), loss = keras.losses.SparseCategoricalCrossentropy(), metrics = ['accuracy'])
  return model

In [56]:
import shutil
shutil.rmtree('/mydir3', ignore_errors=True)

tuner3 = RandomSearch(build_model3, objective = 'val_loss', directory = "/mydir3", max_trials = 5)

In [57]:
tuner3.search(x_train, y_train, epochs = 5, validation_data = (x_test, y_test))

Trial 2 Complete [00h 01m 06s]
val_loss: 2.3019680976867676

Best val_loss So Far: 2.3019680976867676
Total elapsed time: 00h 03m 27s

Search: Running Trial #3

Value             |Best Value So Far |Hyperparameter
3                 |2                 |blocks
128               |64                |kernels0
4                 |3                 |kernel_Size_0
384               |160               |kernels1
4                 |4                 |kernel_Size_1
64                |480               |units
relu              |relu              |act

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 34s 15ms/step - accuracy: 0.1108 - loss: 2.3143 - val_accuracy: 0.1135 - val_loss: 2.3017
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 29s 15ms/step - accuracy: 0.1090 - loss: 2.3027 - val_accuracy: 0.1135 - val_loss: 2.3023
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 38s 14ms/step - accuracy: 0.1084 - loss: 2.3028 - val_accuracy: 0.1135 - val_loss: 2.3014
Epoch 4/5


KeyboardInterrupt: 

In [59]:
model = Sequential(
    [
        CNNBlock(32), CNNBlock(64), CNNBlock(128),
        layers.Flatten(),
        layers.Dense(64, activation = "relu"),
        layers.Dense(10, activation = "softmax")
    ]
)

In [60]:
model.compile(optimizer = keras.optimizers.Adam(), loss = keras.losses.SparseCategoricalCrossentropy(), metrics = ['accuracy'])

In [63]:
checkpoint = keras.callbacks.ModelCheckpoint('/midir2/best.keras', monitor = 'val_loss', mode = 'min', save_best_only=True, save_weights_only=False)

In [64]:
early_stop = keras.callbacks.EarlyStopping(monitor = "val_loss", mode = "min", patience = 5, min_delta = 0.001, verbose = 1)

In [66]:
model.fit(x_train, y_train, epochs = 10, validation_data = (x_test, y_test), callbacks = [checkpoint, early_stop])

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - accuracy: 0.7755 - loss: 0.6563 - val_accuracy: 0.1028 - val_loss: 1400.0251
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9761 - loss: 0.0733 - val_accuracy: 0.1028 - val_loss: 4255.5845
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9844 - loss: 0.0475 - val_accuracy: 0.1028 - val_loss: 111.0355
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9869 - loss: 0.0411 - val_accuracy: 0.1028 - val_loss: 734.6138
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9892 - loss: 0.0317 - val_accuracy: 0.1028 - val_loss: 1896.9082
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9909 - loss: 0.0266 - val_accuracy: 0.0974 - val_loss: 12.5107
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9923 - loss: 0.0235 - val_accuracy: 0.0982 - val_loss: 4.9305
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9946 - 

In [69]:
lr_clipper = keras.callbacks.ReduceLROnPlateau(monitor = "val_loss", mode = "min", patience = 3, min_delta = 0.001, factor = 0.1, verbose = 1, cooldown=2)

In [70]:
model.fit(x_train, y_train, epochs = 10, validation_data = (x_test, y_test), callbacks = [checkpoint, early_stop, lr_clipper])

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9962 - loss: 0.0116 - val_accuracy: 0.1135 - val_loss: 6.5407 - learning_rate: 0.0010
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9961 - loss: 0.0114 - val_accuracy: 0.0980 - val_loss: 4.1787 - learning_rate: 0.0010
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9969 - loss: 0.0091 - val_accuracy: 0.1135 - val_loss: 458.4305 - learning_rate: 0.0010
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9970 - loss: 0.0085 - val_accuracy: 0.0974 - val_loss: 11.1524 - learning_rate: 0.0010
Epoch 5/10
1872/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9965 - loss: 0.0095
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.00010000000474974513.
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9965 - loss: 0.0095 - val_accuracy: 0.1028 - val_loss: 1681.7952 - learning_rate: 0.0010
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.99

In [73]:
class custom_callback(keras.callbacks.Callback):
  def on_end_epoch(self, epoch, logs = None):
    if(logs.get('val_loss') > 0.04):
      print("Training terminated as val_loss exceeded minimum threshold of 0.04")
      self.model.stop_training = True


In [74]:
model.fit(x_train, y_train, epochs = 10, validation_data = (x_test, y_test), callbacks = custom_callback())

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9999 - loss: 6.8732e-04 - val_accuracy: 0.1028 - val_loss: 115.4713
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 1.0000 - loss: 5.3171e-04 - val_accuracy: 0.1028 - val_loss: 330.4655
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 1.0000 - loss: 4.2294e-04 - val_accuracy: 0.0974 - val_loss: 8.1790
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 1.0000 - loss: 3.1217e-04 - val_accuracy: 0.1135 - val_loss: 58.1304
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 1.0000 - loss: 2.7006e-04 - val_accuracy: 0.2569 - val_loss: 17.2299
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 1.0000 - loss: 2.3426e-04 - val_accuracy: 0.4043 - val_loss: 4.4007
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 1.0000 - loss: 1.5974e-04 - val_accuracy: 0.1120 - val_loss: 8.8226
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step -